In [10]:
import pandas as pd
%run ../utils/sampling.py

# Convert CSV files into dataframes
complements_df = pd.read_csv('../data/sample/obj3/sample-complements-top10.csv')
product_df = pd.read_csv('../data/intermediary/product-info.csv')
orders_df = pd.read_csv('../data/intermediary/order-products-enhanced.csv')

C:\Users\jenle\AppData\Local\Temp\ipykernel_8604\1716996786.py:5: DtypeWarning: Columns (0,1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  complements_df = pd.read_csv('../data/sample/obj3/sample-complements-top10.csv')


In [11]:
import numpy as np
from scipy.sparse import csr_matrix
import statsmodels.api as sm

def compute_eci(orders_df, complements_df, product_info, output_csv=None):

    if output_csv:
        with open(output_csv, 'w') as f:
            f.write("product_id,complement_id,coef,p_value,pseudo_r2,ame,eci_units,same_aisle,same_department\n")

    # Map orders and products to indices
    orders_df['order_idx'] = orders_df['order_id'].astype('category').cat.codes
    orders_df['product_idx'] = orders_df['product_id'].astype('category').cat.codes
    product_idx_map = dict(enumerate(orders_df['product_id'].astype('category').cat.categories))
    
    n_orders = orders_df['order_idx'].max() + 1
    n_products = orders_df['product_idx'].max() + 1

    # Sparse basket matrix: rows=orders, cols=products
    data = np.ones(len(orders_df), dtype=np.uint8)
    basket_matrix = csr_matrix((data, (orders_df['order_idx'], orders_df['product_idx'])),
                               shape=(n_orders, n_products))

    # Map product_id -> aisle/department
    aisle_map = product_info.set_index('product_id')['aisle'].to_dict()
    dept_map = product_info.set_index('product_id')['department'].to_dict()

    # Keep unique order-level features
    order_features = orders_df[['order_idx', 'order_size_cat', 'orders_per_user_cat']]\
                        .drop_duplicates('order_idx')\
                        .set_index('order_idx')

    results = []

    for product_id, group in complements_df.groupby('product_id'):

        idx_target = {v: k for k, v in product_idx_map.items()}.get(product_id)
        if idx_target is None:
            continue

        orders_target_idx = set(basket_matrix[:, idx_target].nonzero()[0])
        if not orders_target_idx:
            continue

        for _, row in group.iterrows():
            comp_id = row['complement_id']
            idx_comp = {v: k for k, v in product_idx_map.items()}.get(comp_id)
            if idx_comp is None:
                continue

            y = basket_matrix[:, idx_comp].toarray().ravel()
            target_col = basket_matrix[:, idx_target].toarray().ravel()

            # Build feature matrix using order_features and product similarity
            X = order_features.copy()
            X['target_present'] = target_col
            X['same_aisle'] = int(aisle_map.get(product_id) == aisle_map.get(comp_id))
            X['same_department'] = int(dept_map.get(product_id) == dept_map.get(comp_id))
            X = sm.add_constant(X)

            try:
                model = sm.Logit(y, X).fit(disp=0)
            except:
                continue

            # Compute AME
            X_present = X.copy(); X_present['target_present'] = 1
            X_absent = X.copy(); X_absent['target_present'] = 0
            pred_present = model.predict(X_present)
            pred_absent = model.predict(X_absent)
            ame = (pred_absent - pred_present).mean()

            eci_units = ame * len(y)

            result = {
                'product_id': product_id,
                'complement_id': comp_id,
                'coef': model.params['target_present'],
                'p_value': model.pvalues['target_present'],
                'pseudo_r2': 1 - (model.llf / model.llnull),
                'ame': ame,
                'eci_units': eci_units,
                'same_aisle': int(X['same_aisle'].iloc[0]),
                'same_department': int(X['same_department'].iloc[0])
            }

            results.append(result)

            if output_csv:
                pd.DataFrame([result]).to_csv(output_csv, mode='a', index=False, header=False)

    if not output_csv:
        return pd.DataFrame(results)


#sampled_orders_df = order_sampling_by_num(50000)
complements_df['lift'] = pd.to_numeric(complements_df['lift'], errors='coerce')
top_complements_df = complements_df.nlargest(100, 'lift') 

compute_eci(orders_df, 
    top_complements_df, 
    product_df, 
    output_csv="../data/sample/obj4/top-complements-impact.csv"
)



c:\Users\jenle\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
c:\Users\jenle\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
c:\Users\jenle\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
c:\Users\jenle\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
c:\Users\jenle\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\discrete\discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
c:\Use

In [12]:
# Goodness of fit validation

def validate_goodness_of_fit(df):

    # Threshold-based filtering
    valid = df[(df['p_value'] < 0.05) & (df['pseudo_r2'] > 0.05)]
    print(f"Valid complement pairs: {len(valid)}/{len(df)}")

    # Validation checks
    significant = (df['p_value'] < 0.05).mean()
    avg_r2 = df['pseudo_r2'].mean()

    print(f"{significant:.1%} of models show significant complement effects.")
    print(f"Average pseudo-R²: {avg_r2:.3f}")
    return df

#orders_df = order_sampling_by_num(50000)

#complements_df = pd.read_csv('../data/sample/obj3/sample-complements-top10.csv')
#complements_df['lift'] = pd.to_numeric(complements_df['lift'], errors='coerce')
#top_complements_df = complements_df.nlargest(100, 'lift')

eci_results = pd.read_csv("../data/sample/obj4/top-complements-impact.csv")

validation_df = validate_goodness_of_fit(eci_results)
validation_df.to_csv('../data/sample/obj4/sample-eci-validation.csv', index=False)

Valid complement pairs: 27/27
100.0% of models show significant complement effects.
Average pseudo-R²: inf


In [14]:
# Behavioural validation

def validate_eci_impact(complements_df, test_df):
    # Join test orders to get all product pairs per order
    pairs_in_test = (
        test_df
        .merge(test_df, on="order_id")
        .query("product_id_x != product_id_y")[["product_id_x", "product_id_y"]]
        .drop_duplicates()
    )
    pairs_in_test.columns = ["product_id", "complement_id"]

    # Count observed co-occurrences per complement pair
    observed_counts = (
    test_df
    .merge(complements_df[['product_id', 'complement_id']], on='product_id')
    .merge(test_df, on='order_id', suffixes=('_left', '_right'))
    .query("product_id_right == complement_id")
    .groupby(['product_id_left', 'complement_id'])
    .size()
    .reset_index(name='observed_units')
    .rename(columns={'product_id_left': 'product_id'})
)


    # Merge observed counts with complements_df
    validation_df = complements_df.merge(
        observed_counts,
        on=['product_id', 'complement_id'],
        how='left'
    )
    validation_df['observed_units'] = validation_df['observed_units'].fillna(0)

    # Compute expected units in test using AME × number of test baskets
    num_test_baskets = len(test_df['order_id'].unique())
    validation_df['expected_units_in_test'] = validation_df['ame'] * num_test_baskets

    # Compute error metrics
    validation_df['error_units_test'] = validation_df['expected_units_in_test'] - validation_df['observed_units']

    mse = (validation_df['error_units_test'] ** 2).mean()
    mae = validation_df['error_units_test'].abs().mean()

    print(f"Mean absolute error (units, scaled to test set): {mae:.2f}")
    print(f"Mean squared error (units^2, scaled to test set): {mse:.2f}")

    # Check how many predicted complements appear at least once in test
    num_found = (validation_df['observed_units'] > 0).sum()
    print(f"{num_found} out of {len(validation_df)} predicted complements occurred in test set")

    return validation_df

eci_results = pd.read_csv("../data/sample/obj4/top-complements-impact.csv")
orders_test_df = pd.read_csv('../dataset/order_products__train.csv')

validation_df = validate_eci_impact(eci_results, orders_test_df)
validation_df.to_csv('../data/sample/obj4/sample-eci-validation-behaviour.csv', index=False)

Mean absolute error (units, scaled to test set): 52223.02
Mean squared error (units^2, scaled to test set): 3621242966.10
12 out of 27 predicted complements occurred in test set
